In [ ]:
import gpxpy, gpxpy.gpx, ezdxf
from pyproj import Transformer
import tkinter as tk
from tkinter import filedialog, messagebox
import os

# Descobrir UTM correta com base em coordenadas:
def coordenada_base(lat, lon):
    zona = int((lon + 180) / 6) + 1
    if lat >= 0:
        return 32600 + zona
    else:
        return 32700 + zona

# Converter GPX em DXF normal
def transformar_gpx_em_dxf(gpx_origem, dxf_destino):
    try:
        with open(gpx_origem, "r") as arquivo_gpx:
            gpx = gpxpy.parse(arquivo_gpx)

        if not (gpx.waypoints or gpx.routes or gpx.tracks):
            messagebox.showerror("Erro", "GPX está vazio.")
            return

        # Primeiro ponto para zona UTM
        if gpx.waypoints:
            lat, lon = gpx.waypoints[0].latitude, gpx.waypoints[0].longitude
        elif gpx.routes and gpx.routes[0].points:
            lat, lon = gpx.routes[0].points[0].latitude, gpx.routes[0].points[0].longitude
        else:
            lat, lon = gpx.tracks[0].segments[0].points[0].latitude, gpx.tracks[0].segments[0].longitude

        codigo_utm = coordenada_base(lat, lon)
        transformar = Transformer.from_crs("EPSG:4326", f"EPSG:{codigo_utm}", always_xy=True)

        doc = ezdxf.new("R2007")
        msp = doc.modelspace()

        def transformar_ponto(p):
            return transformar.transform(p.longitude, p.latitude)

        for ponto in gpx.waypoints:
            tamanho = 3
            x, y = transformar_ponto(ponto)
            msp.add_line((x - tamanho, y - tamanho), (x + tamanho, y + tamanho), dxfattribs={'color': 3})
            msp.add_line((x - tamanho, y + tamanho), (x + tamanho, y - tamanho), dxfattribs={'color': 3})
            if ponto.name:
                msp.add_text(ponto.name, dxfattribs={'height': 10}).set_placement((x, y, 0))

        doc.saveas(dxf_destino)
        messagebox.showinfo("Sucesso", f"Arquivo DXF criado em:\n{dxf_destino}")

    except Exception as e:
        messagebox.showerror("Erro", f"Ocorreu um erro durante a conversão:\n{e}")

# Converter GPX em DXF usando ponto de referência
def transformar_gpx_em_dxf_coordenado(gpx_origem, dxf_destino, x_ref, y_ref):
    try:
        with open(gpx_origem, "r") as arquivo_gpx:
            gpx = gpxpy.parse(arquivo_gpx)

        if not gpx.waypoints:
            messagebox.showerror("Erro", "GPX sem pontos válidos.")
            return

        lat0, lon0 = gpx.waypoints[0].latitude, gpx.waypoints[0].longitude
        codigo_utm = coordenada_base(lat0, lon0)
        transformar = Transformer.from_crs("EPSG:4326", f"EPSG:{codigo_utm}", always_xy=True)

        #Cria o desenho DXF
        doc = ezdxf.new("R2007")
        msp = doc.modelspace()

        def convert(n):
            return float(str(n)[:10])  # limita a 10 dígitos
        

        def transformar_ponto(p):
            # deslocamento
            rx, ry = transformar.transform(gpx.waypoints[0].longitude, gpx.waypoints[0].latitude)
            dx, dy = x_ref - convert(rx), y_ref - convert(ry)
            x, y = transformar.transform(p.longitude, p.latitude)
            return convert(x) + dx, convert(y) + dy
        

        for ponto in gpx.waypoints:
            tamanho = 3
            x, y = transformar_ponto(ponto)
            msp.add_line((x - tamanho, y - tamanho), (x + tamanho, y + tamanho), dxfattribs={'color': 3})
            msp.add_line((x - tamanho, y + tamanho), (x + tamanho, y - tamanho), dxfattribs={'color': 3})
            if ponto.name:
                msp.add_text(ponto.name, dxfattribs={'height': 10}).set_placement((x, y, 0))

        doc.saveas(dxf_destino)
        messagebox.showinfo("Sucesso", f"Arquivo DXF criado em:\n{dxf_destino}")

    except Exception as e:
        messagebox.showerror("Erro", f"Ocorreu um erro durante a conversão:\n{e}")

# --- INTERFACE TKINTER ---

janela = tk.Tk()
janela.title("Conversor GPX/DXF")
janela.geometry("500x380")

gpx_origem = None
dxf_destino = None

def escolher_arquivo():
    global gpx_origem, dxf_destino
    gpx_origem = filedialog.askopenfilename(filetypes=[("Arquivos GPX", "*.gpx")])
    if gpx_origem:
        lbl_arquivo.config(text=f"Selecionado: {gpx_origem}")

        # sugere o mesmo nome trocando extensão
        nome_base = os.path.splitext(os.path.basename(gpx_origem))[0] + ".dxf"
        pasta = os.path.dirname(gpx_origem)

        dxf_destino = filedialog.asksaveasfilename(
            defaultextension=".dxf",
            initialfile=nome_base,
            initialdir=pasta,
            filetypes=[("DXF files", "*.dxf")]
        )

        if dxf_destino:
            lbl_destino.config(text=f"Salvar em: {dxf_destino}")
        else:
            lbl_destino.config(text="Nenhum destino selecionado")
    else:
        lbl_arquivo.config(text="Nenhum arquivo selecionado")

def converter():
    if not gpx_origem:
        messagebox.showerror("Erro", "Selecione um arquivo GPX primeiro!")
        return
    if not dxf_destino:
        messagebox.showerror("Erro", "Escolha o local de destino do DXF!")
        return

    if var_ref.get() == 1:  # com ponto de referência
        try:
            x_ref = float(entry_x.get())
            y_ref = float(entry_y.get())
            transformar_gpx_em_dxf_coordenado(gpx_origem, dxf_destino, x_ref, y_ref)
        except ValueError:
            messagebox.showerror("Erro", "Digite valores numéricos válidos para X e Y.")
    else:  # normal
        transformar_gpx_em_dxf(gpx_origem, dxf_destino)

# Botões e labels
btn_arquivo = tk.Button(janela, text="Selecionar arquivo GPX e destino", command=escolher_arquivo)
btn_arquivo.pack(pady=5)

lbl_arquivo = tk.Label(janela, text="Nenhum arquivo selecionado")
lbl_arquivo.pack()

lbl_destino = tk.Label(janela, text="Nenhum destino selecionado")
lbl_destino.pack()

var_ref = tk.IntVar()
chk_ref = tk.Checkbutton(janela, text="Usar ponto de referência", variable=var_ref)
chk_ref.pack(pady=5)

frame_ref = tk.Frame(janela)
tk.Label(frame_ref, text="X:").grid(row=0, column=0, padx=5)
entry_x = tk.Entry(frame_ref, width=12)
entry_x.grid(row=0, column=1)

tk.Label(frame_ref, text="Y:").grid(row=0, column=2, padx=5)
entry_y = tk.Entry(frame_ref, width=12)
entry_y.grid(row=0, column=3)
frame_ref.pack()

btn_converter = tk.Button(janela, text="Converter para DXF", command=converter)
btn_converter.pack(pady=20)

janela.mainloop()


In [2]:
import gpxpy, gpxpy.gpx, ezdxf
from pyproj import Transformer

def coordenada_base(lat, lon):
    zona = int((lon + 180) / 6) + 1
    if lat >= 0:
        return 32600 + zona
    else:
        return 32700 + zona
    
def transformar_gpx_em_dxf(gpx_origem, dxf_destino):
    try:
        with open(gpx_origem, "r") as arquivo_gpx:
            gpx = gpxpy.parse(arquivo_gpx)

        if not (gpx.waypoints or gpx.routes or gpx.tracks):
            print("Erro", "GPX está vazio.")
            return

        # Primeiro ponto para zona UTM
        if gpx.waypoints:
            lat, lon = gpx.waypoints[0].latitude, gpx.waypoints[0].longitude
        elif gpx.routes and gpx.routes[0].points:
            lat, lon = gpx.routes[0].points[0].latitude, gpx.routes[0].points[0].longitude
        else:
            lat, lon = gpx.tracks[0].segments[0].points[0].latitude, gpx.tracks[0].segments[0].longitude

        codigo_utm = coordenada_base(lat, lon)
        transformar = Transformer.from_crs("EPSG:4326", f"EPSG:{codigo_utm}", always_xy=True)

        doc = ezdxf.new("R2007")
        msp = doc.modelspace()

        def transformar_ponto(p):
            return transformar.transform(p.longitude, p.latitude)
        
        pontos_nome = { }

        for ponto in gpx.waypoints:
            x, y = transformar_ponto(ponto)
            pontos_nome[ponto.name] = x, y

            tamanho = 3
            x, y = transformar_ponto(ponto)
            msp.add_line((x - tamanho, y - tamanho), (x + tamanho, y + tamanho), dxfattribs={'color': 3})
            msp.add_line((x - tamanho, y + tamanho), (x + tamanho, y - tamanho), dxfattribs={'color': 3})
            if ponto.name:
                msp.add_text(ponto.name, dxfattribs={'height': 10}).set_placement((x, y, 0))
        
        print(pontos_nome)

        doc.saveas(dxf_destino)
    except Exception as e:
        print(f"erro: {e}")
        

gpx_origem = "C:\\Users\\Wellison\\Desktop\\teste\\arquivo.gpx"
dxf_destino = "C:\\Users\\Wellison\\Desktop\\teste\\arquivo.dxf"

transformar_gpx_em_dxf(gpx_origem, dxf_destino)





{'001': (577787.3849670856, 9367336.178628819), '002': (577786.942443946, 9367336.510818284), '003': (577750.2136833497, 9367364.635277493), '004': (577713.4844862331, 9367392.428063473), '005': (577678.7488443167, 9367420.550038343), '006': (559750.6509714582, 9397596.393385561), '007': (559750.98291667, 9397595.950908817), '008': (559752.2008068474, 9397595.176004976), '009': (559771.2512419261, 9397590.958259296), '010': (543041.5955456722, 9413933.695594441), '011': (543041.4847411249, 9413933.69566386), '012': (543075.9580554438, 9413954.56651175), '013': (543102.8988269718, 9413978.868874425)}


In [4]:
def transformar_gpx_em_dxf_coordenado(gpx_origem, dxf_destino, x_ref, y_ref, ref):
    try:
        with open(gpx_origem, "r") as arquivo_gpx:
            gpx = gpxpy.parse(arquivo_gpx)

        if not gpx.waypoints:
            print("Erro", "GPX sem pontos válidos.")
            return

        lat0, lon0 = gpx.waypoints[0].latitude, gpx.waypoints[0].longitude
        codigo_utm = coordenada_base(lat0, lon0)
        transformar = Transformer.from_crs("EPSG:4326", f"EPSG:{codigo_utm}", always_xy=True)

        def convert(n):
            return float(str(n)[:10])  # limita a 10 dígitos

        doc = ezdxf.new("R2007")
        msp = doc.modelspace()

        def diferenca(ref, x_ref, y_ref):
            nome_ponto = {}
            for ponto in gpx.waypoints:
                x, y = transformar.transform(ponto.longitude, ponto.latitude)
                nome_ponto[ponto.name] = x, y
            x = nome_ponto[ref][0]
            y = nome_ponto[ref][1]
            dx = x_ref - x
            dy = y_ref - y

            return dx, dy
        

        def transformar_ponto(p):
            x, y = transformar.transform(p.longitude, p.latitude)
            dx, dy = diferenca(ref, x_ref, y_ref)
            return convert(x) + dx, convert(y)+ dy 

        for ponto in gpx.waypoints:
            tamanho = 3
            x, y = transformar_ponto(ponto)
            msp.add_line((x - tamanho, y - tamanho), (x + tamanho, y + tamanho), dxfattribs={'color': 3})
            msp.add_line((x - tamanho, y + tamanho), (x + tamanho, y - tamanho), dxfattribs={'color': 3})
            if ponto.name:
                msp.add_text(ponto.name, dxfattribs={'height': 10}).set_placement((x, y, 0))
            print(x, y)

    

        doc.saveas(dxf_destino)
        print("Sucesso", f"Arquivo DXF criado em:\n{dxf_destino}")

    except Exception as e:
        print("Erro", f"Ocorreu um erro durante a conversão:\n{e}")

gpx_origem = "C:\\Users\\Wellison\\Desktop\\teste\\arquivo.gpx"
dxf_destino = "C:\\Users\\Wellison\\Desktop\\teste\\arquivo.dxf"
x_ref = 543109.913 
y_ref = 9414000.23
ref = "010"

transformar_gpx_em_dxf_coordenado(gpx_origem, dxf_destino, x_ref, y_ref, ref)


577855.7014543277 9367402.70440556
577855.2594543278 9367403.04440556
577855.2594543278 9367403.04440556
577818.5304543277 9367431.16440556
577818.5304543277 9367431.16440556
577781.8014543278 9367458.95440556
577781.8014543278 9367458.95440556
577747.0654543277 9367487.08440556
577747.0654543277 9367487.08440556
559818.9674543277 9397662.92440556
559819.2994543277 9397662.484405559
559820.5174543277 9397661.70440556
559820.5174543277 9397661.70440556
559839.5684543278 9397657.484405559
559839.5684543278 9397657.484405559
543109.9124543277 9414000.224405559
543109.8014543278 9414000.224405559
543109.8014543278 9414000.224405559
543144.2754543277 9414021.09440556
543144.2754543277 9414021.09440556
543171.2154543278 9414045.394405559
543171.2154543278 9414045.394405559
Sucesso Arquivo DXF criado em:
C:\Users\Wellison\Desktop\teste\arquivo.dxf


In [11]:
import os

gpx_file = "C:\\Users\\Wellison\\Desktop\\teste\\gpx\\arquivo.gpx"
print(os.path.exists(gpx_file))


True
